In [0]:
%pip install praw

dbutils.library.restartPython()

In [0]:
#Populating the Main Table
#Overwriting the tables not appending
import re
import praw
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, lit, to_date, explode, split, trim, length, regexp_replace, udf
from datetime import datetime, timezone

spark = SparkSession.builder.appName("BeautyTalkPH_Initial").getOrCreate()

# ── Credentials ───────────────────────────────────────────────────────────────
CLIENT_ID = dbutils.secrets.get(
    scope="reddit-secrets",
    key="client-id"
)

CLIENT_SECRET = dbutils.secrets.get(
    scope="reddit-secrets",
    key="client-secret"
)

USER_AGENT = dbutils.secrets.get(
    scope="reddit-secrets",
    key="user-agent"
)

reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT,
)

# ── Config ────────────────────────────────────────────────────────────────────
SUBREDDIT       = "beautytalkph"
DELTA_DATABASE  = "beauty_analytics"
POSTS_TABLE     = f"{DELTA_DATABASE}.posts"
COMMENTS_TABLE  = f"{DELTA_DATABASE}.comments"
POST_SENT_TABLE = f"{DELTA_DATABASE}.post_sentences"
COMM_SENT_TABLE = f"{DELTA_DATABASE}.comment_sentences"
BRANDS_TABLE    = f"{DELTA_DATABASE}.brands"
FLAIR_FILTER    = "review"
TOP_LIMIT       = 2000

# ── Load brands ───────────────────────────────────────────────────────────────
spark.sql(f"CREATE DATABASE IF NOT EXISTS {DELTA_DATABASE}")

local_brands = [
    "ever bilena", "blk", "colourette", "filipinta beauty", "lip pinas",
    "clochefame", "cloud", "beach born", "sunnies", "belo",
    "happy skin", "issy", "lucky beauty", "popique",
    "dr. sensitive", "grwm", "teviant", "avon", "vice", "enigma",
    "ellana", "careline", "shawill", "miniso", "kkv",
    "chuchu", "detail", "dazzle me", "strokes", "time phoria",
    "cloudbeauty", "rmt", "imma", "nichido", "absidy", "squad"
]

international_brands = [
    "3ce", "apieu", "armani", "barenbliss", "bbia", "benefit", "canmake",
    "carslan", "catkin", "cacharel", "carolina herrera", "cezanne",
    "charlotte tilbury", "chifure", "clio", "clinique", "colorkey",
    "colourpop", "dasique", "dior", "elf", "espoir", "estee lauder",
    "etude house", "excel", "fenty", "florasis", "flower knows", "focallure",
    "gisou", "glossier", "heimish", "hold live", "holika holika", "hoola",
    "huda beauty", "hince", "into you", "jmcy", "judydoll", "kate", "kiko",
    "kiss me", "kose", "kosas", "kylie", "laura mercier", "laneige",
    "lancome", "loreal", "luxe cosmetics", "mac", "magefy", "maange",
    "majolica majorca", "make up for ever", "mao ge ping", "marc jacobs",
    "mario", "maybelline", "milk", "missha", "monotheme", "moonshot",
    "morphe", "nars", "opera", "otwoo", "patrick ta", "perfect diary",
    "peripera", "pinkflash", "rare beauty", "revlon", "rhode", "romand",
    "sace lady", "sasi", "sea makeup", "shiseido", "sheglam",
    "summer fridays", "the saem", "tony moly", "too faced", "tower 28",
    "vdl", "visee", "ysl", "zenn"
]

brands_list = local_brands + international_brands

# ── Keyword Lists ─────────────────────────────────────────────────────────────
makeup_categories = [
    "foundation", "concealer", "primer", "powder", "blush", "highlighter",
    "bronzer", "contour", "setting spray", "fixing spray", "eyeshadow",
    "eyeliner", "mascara", "eyebrow", "brow gel", "brow pencil", "lash",
    "eye primer", "palette", "lipstick", "lip gloss", "lip liner", "lip balm",
    "lip oil", "lip plumper", "lip tint", "lip lacquer", "lip stain",
    "skin tint", "brush", "sponge", "eyelash curler", "tweezer",
    "makeup remover", "bb cream", "cc cream", "dd cream", "moisturizer",
    "color corrector", "multistick", "sunscreen"
]

positive_keywords = [
    "love", "loved", "great", "amazing", "excellent", "awesome", "perfect",
    "good", "nice", "best", "beautiful", "fantastic", "recommend", "worth it",
    "holy grail", "repurchase", "favorite", "favourite", "impressed",
    "stunning", "flawless", "lightweight", "comfortable", "long lasting",
    "long-lasting", "affordable", "value", "works", "effective", "happy",
    "obsessed", "glowy", "natural", "smooth", "soft", "hydrating",
    "maganda", "magaling", "ganda", "galing", "sulit", "bet", "winner",
    "solid", "grabeh", "grabe", "astig", "okay siya", "ok siya",
    "nagustuhan", "gusto ko", "love na love", "swak", "bagay", "ayos",
    "magaan", "maayos", "worth", "worth it talaga",
]

negative_keywords = [
    "bad", "worst", "terrible", "awful", "hate", "dislike", "disappointed",
    "disappointing", "poor", "waste", "overpriced", "expensive", "cakey",
    "creasy", "crease", "oxidize", "oxidized", "patchy", "streaky",
    "heavy", "greasy", "oily", "dry", "drying", "irritated", "irritating",
    "broke out", "breakout", "pimple", "allergy", "allergic",
    "not worth", "would not recommend", "return", "fake", "expired",
    "hindi maganda", "hindi okay", "hindi ok", "hindi sulit", "mahal",
    "pangit", "basura", "hindi bet", "nakaka-off", "nakakainis",
    "nakaka-irritate", "sumabog", "nag-oxidize", "hindi maayos",
    "peke", "bigo", "nabigo", "hindi worth it", "sayang", "ayaw ko",
    "di maganda", "di okay", "di ok", "di sulit", "di bet",
]

# ── Pre-compile regex patterns ────────────────────────────────────────────────
# Compiling once up front is much faster than re.search inside a loop each call
brands_patterns    = [(b, re.compile(r'\b' + re.escape(b) + r'\b')) for b in brands_list]
category_patterns  = [(c, re.compile(r'\b' + re.escape(c) + r'\b')) for c in makeup_categories]
platform_patterns  = [(p, re.compile(r'\b' + re.escape(p) + r'\b')) for p in ["tiktok", "shopee", "lazada"]]
positive_patterns  = [re.compile(r'\b' + re.escape(k) + r'\b') for k in positive_keywords]
negative_patterns  = [re.compile(r'\b' + re.escape(k) + r'\b') for k in negative_keywords]

# ── Helper Functions ──────────────────────────────────────────────────────────
def detect_brands(text):
    if not text:
        return None
    t = text.lower()
    found = [b for b, pat in brands_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_category(text):
    if not text:
        return None
    t = text.lower()
    found = [c for c, pat in category_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_platforms(text):
    if not text:
        return None
    t = text.lower()
    found = [p for p, pat in platform_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_sentiment(text):
    if not text:
        return "neutral"
    t = text.lower()
    pos = sum(1 for pat in positive_patterns if pat.search(t))
    neg = sum(1 for pat in negative_patterns if pat.search(t))
    if pos > neg:   return "positive"
    elif neg > pos: return "negative"
    else:           return "neutral"

def ts_to_date(utc_ts):
    return datetime.utcfromtimestamp(utc_ts).strftime("%Y-%m-%d")

detect_brands_udf    = udf(detect_brands,    StringType())
detect_category_udf  = udf(detect_category,  StringType())
detect_platforms_udf = udf(detect_platforms, StringType())
detect_sentiment_udf = udf(detect_sentiment, StringType())

# ── Scrape ALL historical posts using top() ───────────────────────────────────
subreddit     = reddit.subreddit(SUBREDDIT)
posts_rows    = []
comments_rows = []

print("🔍 Scraping all historical posts...")

for submission in subreddit.top(limit=TOP_LIMIT):
    flair = submission.link_flair_text
    if not flair or FLAIR_FILTER not in flair.lower():
        continue

    post_content = submission.title
    if submission.selftext:
        post_content += " " + submission.selftext

    posts_rows.append((
        submission.id,
        post_content,
        submission.url,
        int(submission.score),
        int(submission.num_comments),
        ts_to_date(submission.created_utc),
    ))

    submission.comments.replace_more(limit=0)
    for comment in submission.comments.list():
        comments_rows.append((
            submission.id,
            comment.id,
            comment.body,
            int(comment.score),
            ts_to_date(comment.created_utc),
        ))

print(f"✅ Found {len(posts_rows)} posts, {len(comments_rows)} comments.")

# ── Schemas ───────────────────────────────────────────────────────────────────
posts_schema = StructType([
    StructField("post_id",            StringType(),  False),
    StructField("post_content",       StringType(),  True),
    StructField("post_url",           StringType(),  True),
    StructField("number_of_upvotes",  IntegerType(), True),
    StructField("number_of_comments", IntegerType(), True),
    StructField("date",               StringType(),  True),
])

comments_schema = StructType([
    StructField("post_id",         StringType(),  False),
    StructField("comment_id",      StringType(),  False),
    StructField("comment_body",    StringType(),  True),
    StructField("comment_upvotes", IntegerType(), True),
    StructField("date",            StringType(),  True),
])

# ── Write with overwrite (initial load) ───────────────────────────────────────
posts_df    = spark.createDataFrame(posts_rows,    schema=posts_schema)
comments_df = spark.createDataFrame(comments_rows, schema=comments_schema)

posts_df    = posts_df.withColumn("date",    to_date(col("date"), "yyyy-MM-dd"))
comments_df = comments_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

posts_df = posts_df \
    .withColumn("brands_mentioned",   detect_brands_udf(col("post_content"))) \
    .withColumn("category_mentioned", detect_category_udf(col("post_content"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("post_content")))

comments_df = comments_df \
    .withColumn("brands_mentioned",   detect_brands_udf(col("comment_body"))) \
    .withColumn("category_mentioned", detect_category_udf(col("comment_body"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("comment_body")))

posts_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(POSTS_TABLE)

comments_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMMENTS_TABLE)

print(f"✅ Posts saved:    {posts_df.count()} rows")
print(f"✅ Comments saved: {comments_df.count()} rows")

# ── Post Sentence Table ───────────────────────────────────────────────────────
print("🔍 Building post sentences...")

post_sentences_df = spark.table(POSTS_TABLE) \
    .select("post_id", "post_content") \
    .withColumn("post_content", regexp_replace(col("post_content"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("post_content"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("post_content") \
    .dropDuplicates(["post_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull())

post_sentences_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(POST_SENT_TABLE)

print(f"✅ Post sentences saved: {post_sentences_df.count()} rows")

# ── Comment Sentence Table ────────────────────────────────────────────────────
print("🔍 Building comment sentences...")

comment_sentences_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "comment_body") \
    .withColumn("comment_body", regexp_replace(col("comment_body"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("comment_body"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("comment_body") \
    .dropDuplicates(["comment_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull())

comment_sentences_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMM_SENT_TABLE)

print(f"✅ Comment sentences saved: {comment_sentences_df.count()} rows")

print("\n🎉 Initial load complete!")
print(f"   Posts:             {spark.table(POSTS_TABLE).count()} total rows")
print(f"   Comments:          {spark.table(COMMENTS_TABLE).count()} total rows")
print(f"   Post sentences:    {spark.table(POST_SENT_TABLE).count()} total rows")
print(f"   Comment sentences: {spark.table(COMM_SENT_TABLE).count()} total rows")

In [0]:
#Populating the dimension tables (smaller) using the data of Post Table
#Unpivoting the columns
#Overwriting the tables
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, trim, when

spark = SparkSession.builder.appName("BeautyTalkPH_Unpivot").getOrCreate()

# ── Config ────────────────────────────────────────────────────────────────────
DELTA_DATABASE         = "beauty_analytics"
POSTS_TABLE            = f"{DELTA_DATABASE}.posts"
BRANDS_UNPIVOT_TABLE   = f"{DELTA_DATABASE}.post_brands_mentioned"
CATEGORY_UNPIVOT_TABLE = f"{DELTA_DATABASE}.post_categories_mentioned"
PLATFORM_UNPIVOT_TABLE = f"{DELTA_DATABASE}.post_platforms_mentioned"

# ── Brand Lists ───────────────────────────────────────────────────────────────
local_brands = [
    "ever bilena", "blk", "colourette", "filipinta beauty", "lip pinas",
    "clochefame", "cloud", "beach born", "sunnies", "belo",
    "happy skin", "issy", "lucky beauty", "popique",
    "dr. sensitive", "grwm", "teviant", "avon", "vice", "enigma",
    "ellana", "careline",  "shawill", "miniso", "kkv",

    # added
    "chuchu",
    "detail",
    "dazzle me",
    "strokes",
    "time phoria",
    "cloudbeauty",
    "rmt",
    "imma",
    "nichido",
    "absidy", "squad"
]

international_brands = ["3ce", "apieu", "armani", "barenbliss", "bbia", "benefit", "canmake", "carslan", "catkin", "cacharel", "carolina herrera", "cezanne", "charlotte tilbury", "chifure", "clio", "clinique", "colorkey", "colourpop", "dasique", "dior", "elf", "espoir", "estee lauder", "etude house", "excel", "fenty", "florasis", "flower knows", "focallure", "gisou", "glossier", "heimish", "hold live", "holika holika", "hoola", "huda beauty", "hince", "into you", "jmcy", "judydoll", "kate", "kiko", "kiss me", "kose", "kosas", "kylie", "laura mercier", "laneige", "lancome", "loreal", "luxe cosmetics", "mac", "magefy", "maange", "majolica majorca", "make up for ever", "mao ge ping", "marc jacobs", "mario", "maybelline", "milk", "missha", "monotheme", "moonshot", "morphe", "nars", "opera", "otwoo", "patrick ta", "perfect diary", "peripera", "pinkflash", "rare beauty", "revlon", "rhode", "romand", "sace lady", "sasi", "sea makeup", "shiseido", "sheglam", "summer fridays", "the saem", "tony moly", "too faced", "tower 28", "vdl", "visee", "ysl", "zenn"]

# ── Unpivot Brands Mentioned ──────────────────────────────────────────────────
print("🔍 Processing brands mentioned...")

brands_mentioned_df = spark.table(POSTS_TABLE) \
    .select("post_id", "brands_mentioned") \
    .filter(col("brands_mentioned").isNotNull()) \
    .withColumn("brand", explode(split(col("brands_mentioned"), ","))) \
    .withColumn("brand", trim(col("brand"))) \
    .withColumn("origin", when(col("brand").isin(local_brands), "local")
                          .when(col("brand").isin(international_brands), "international")
                          .otherwise("unknown")) \
    .select("post_id", "brand", "origin") \
    .dropDuplicates()

brands_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(BRANDS_UNPIVOT_TABLE)

print(f"✅ Brands mentioned saved: {brands_mentioned_df.count()} rows")

# ── Unpivot Categories Mentioned ──────────────────────────────────────────────
print("🔍 Processing categories mentioned...")

categories_mentioned_df = spark.table(POSTS_TABLE) \
    .select("post_id", "category_mentioned") \
    .filter(col("category_mentioned").isNotNull()) \
    .withColumn("category", explode(split(col("category_mentioned"), ","))) \
    .withColumn("category", trim(col("category"))) \
    .select("post_id", "category") \
    .dropDuplicates()

categories_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(CATEGORY_UNPIVOT_TABLE)

print(f"✅ Categories mentioned saved: {categories_mentioned_df.count()} rows")

# ── Unpivot Platforms Mentioned ───────────────────────────────────────────────
print("🔍 Processing platforms mentioned...")

platforms_mentioned_df = spark.table(POSTS_TABLE) \
    .select("post_id", "platform_mentioned") \
    .filter(col("platform_mentioned").isNotNull()) \
    .withColumn("platform", explode(split(col("platform_mentioned"), ","))) \
    .withColumn("platform", trim(col("platform"))) \
    .select("post_id", "platform") \
    .dropDuplicates()

platforms_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(PLATFORM_UNPIVOT_TABLE)

print(f"✅ Platforms mentioned saved: {platforms_mentioned_df.count()} rows")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n🎉 Unpivot complete!")
print(f"   Brands:     {spark.table(BRANDS_UNPIVOT_TABLE).count()} total rows")
print(f"   Categories: {spark.table(CATEGORY_UNPIVOT_TABLE).count()} total rows")
print(f"   Platforms:  {spark.table(PLATFORM_UNPIVOT_TABLE).count()} total rows")

In [0]:
#Populating the dimension tables (smaller) using the data of Comment Table
#Unpivoting the columns
#Overwriting the tables
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, split, explode, trim

spark = SparkSession.builder.appName("BeautyTalkPH_Comments_Unpivot").getOrCreate()

# ── Config ────────────────────────────────────────────────────────────────────
DELTA_DATABASE  = "beauty_analytics"
COMMENTS_TABLE  = f"{DELTA_DATABASE}.comments"
COMMENT_BRANDS_UNPIVOT_TABLE = f"{DELTA_DATABASE}.comment_brands_mentioned"
COMMENT_CATEGORY_UNPIVOT_TABLE = f"{DELTA_DATABASE}.comment_categories_mentioned"
COMMENT_PLATFORM_UNPIVOT_TABLE = f"{DELTA_DATABASE}.comment_platforms_mentioned"

# ── Unpivot Brands Mentioned ──────────────────────────────────────────────────
print("🔍 Processing comment brands mentioned...")

comment_brands_mentioned_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "brands_mentioned") \
    .filter(col("brands_mentioned").isNotNull()) \
    .withColumn("brand", explode(split(col("brands_mentioned"), ","))) \
    .withColumn("brand", trim(col("brand"))) \
    .select("post_id", "comment_id", "brand") \
    .dropDuplicates()

comment_brands_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMMENT_BRANDS_UNPIVOT_TABLE)

print(f"✅ Comment brands mentioned saved: {comment_brands_mentioned_df.count()} rows")

# ── Unpivot Categories Mentioned ──────────────────────────────────────────────
print("🔍 Processing comment categories mentioned...")

comment_categories_mentioned_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "category_mentioned") \
    .filter(col("category_mentioned").isNotNull()) \
    .withColumn("category", explode(split(col("category_mentioned"), ","))) \
    .withColumn("category", trim(col("category"))) \
    .select("post_id", "comment_id", "category") \
    .dropDuplicates()

comment_categories_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMMENT_CATEGORY_UNPIVOT_TABLE)

print(f"✅ Comment categories mentioned saved: {comment_categories_mentioned_df.count()} rows")

# ── Unpivot Platforms Mentioned ───────────────────────────────────────────────
print("🔍 Processing comment platforms mentioned...")

comment_platforms_mentioned_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "platform_mentioned") \
    .filter(col("platform_mentioned").isNotNull()) \
    .withColumn("platform", explode(split(col("platform_mentioned"), ","))) \
    .withColumn("platform", trim(col("platform"))) \
    .select("post_id", "comment_id", "platform") \
    .dropDuplicates()

comment_platforms_mentioned_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMMENT_PLATFORM_UNPIVOT_TABLE)

print(f"✅ Comment platforms mentioned saved: {comment_platforms_mentioned_df.count()} rows")

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n🎉 Comment unpivot complete!")
print(f"   Comment Brands:     {spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).count()} total rows")
print(f"   Comment Categories: {spark.table(COMMENT_CATEGORY_UNPIVOT_TABLE).count()} total rows")
print(f"   Comment Platforms:  {spark.table(COMMENT_PLATFORM_UNPIVOT_TABLE).count()} total rows")

# ── Display Tables ────────────────────────────────────────────────────────────
print("\n" + "="*80)
print("📊 COMMENT BRANDS MENTIONED TABLE")
print("="*80)
spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).show(20, truncate=False)

print("\n" + "="*80)
print("📊 COMMENT CATEGORIES MENTIONED TABLE")
print("="*80)
spark.table(COMMENT_CATEGORY_UNPIVOT_TABLE).show(20, truncate=False)

print("\n" + "="*80)
print("📊 COMMENT PLATFORMS MENTIONED TABLE")
print("="*80)
spark.table(COMMENT_PLATFORM_UNPIVOT_TABLE).show(20, truncate=False)

# ── Display with different options ────────────────────────────────────────────
# Option 1: Show first 10 rows (default is 20)
# spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).show(10)

# Option 2: Show all rows (be careful with large datasets!)
# spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).show(spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).count(), truncate=False)

# Option 3: Display as a nice formatted string
# print(spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).toPandas().to_string())

# Option 4: Get schema information
# spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).printSchema()

# Option 5: Display with vertical orientation (useful for wide tables)
# spark.table(COMMENT_BRANDS_UNPIVOT_TABLE).show(10, vertical=True)

In [0]:
#Determining the Sentinments of each sentence for both post and comments table
#Overwrites the table
import re
from typing import List, Tuple
import praw
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, IntegerType
from pyspark.sql.functions import col, lit, to_date, explode, split, trim, length, regexp_replace, udf
from datetime import datetime, timedelta, timezone

spark = SparkSession.builder.appName("BeautyTalkPH_Weekly").getOrCreate()

# ── Credentials ───────────────────────────────────────────────────────────────
CLIENT_ID     = "b2fte9KPWGuMmYNH228_1w"
CLIENT_SECRET = "29Kz2dO1m-pvg26h8ROcAs-neh0v6A"
USER_AGENT    = "beautytalkph-scraper/0.1 by u/flawedhumannn"

reddit = praw.Reddit(
    client_id=CLIENT_ID,
    client_secret=CLIENT_SECRET,
    user_agent=USER_AGENT,
)

# ── Config ────────────────────────────────────────────────────────────────────
SUBREDDIT       = "beautytalkph"
DELTA_DATABASE  = "beauty_analytics"
POSTS_TABLE     = f"{DELTA_DATABASE}.posts"
COMMENTS_TABLE  = f"{DELTA_DATABASE}.comments"
POST_SENT_TABLE = f"{DELTA_DATABASE}.post_sentences"
COMM_SENT_TABLE = f"{DELTA_DATABASE}.comment_sentences"
BRANDS_TABLE    = f"{DELTA_DATABASE}.brands"
FLAIR_FILTER    = "review"

# ── Last 7 days only ──────────────────────────────────────────────────────────
DATE_TO   = datetime.now(timezone.utc)
DATE_FROM = DATE_TO - timedelta(days=7)

print(f"📅 Scraping from {DATE_FROM.strftime('%Y-%m-%d')} to {DATE_TO.strftime('%Y-%m-%d')}")

# ── Load brands ───────────────────────────────────────────────────────────────
brands_list: List[str] = [row["brand"] for row in spark.table(BRANDS_TABLE).collect() if row["brand"] is not None]

# ── Keyword Lists ─────────────────────────────────────────────────────────────
makeup_categories = [
    "foundation", "foundations", "concealer", "concealers",
    "primer", "primers", "powder", "powders",
    "blush", "blushes", "highlighter", "highlighters",
    "bronzer", "bronzers", "contour", "contours",
    "setting spray", "setting sprays", "fixing spray", "fixing sprays",
    "eyeshadow", "eyeshadows", "eyeliner", "eyeliners",
    "mascara", "mascaras", "eyebrow", "brow gel", "brow",
    "lashes", "lash", "eye primer", "eye primers", "palette", "palettes",
    "lipstick", "lipsticks", "lip gloss", "lip glosses",
    "lip liner", "lip liners", "lip balm", "lip balms",
    "lip stain", "lip stains", "skin tint", "skin tints",
    "brush", "brushes", "sponge", "sponges",
    "eyelash curler", "eyelash curlers", "tweezers", "tweezer",
    "makeup remover", "makeup removers",
    "bb cream", "bb creams", "cc cream", "cc creams", "dd cream", "dd creams",
    "moisturizer", "moisturizers", "makeup",
    "color corrector", "color correctors",
]

positive_keywords = [
    "love", "loved", "great", "amazing", "excellent", "awesome", "perfect",
    "good", "nice", "best", "beautiful", "fantastic", "recommend", "worth it",
    "holy grail", "repurchase", "favorite", "favourite", "impressed",
    "stunning", "flawless", "lightweight", "comfortable", "long lasting",
    "long-lasting", "affordable", "value", "works", "effective", "happy",
    "obsessed", "glowy", "natural", "smooth", "soft", "hydrating",
    "maganda", "magaling", "ganda", "galing", "sulit", "bet", "winner",
    "solid", "grabeh", "grabe", "astig", "okay siya", "ok siya",
    "nagustuhan", "gusto ko", "love na love", "swak", "bagay", "ayos",
    "magaan", "maayos", "worth", "worth it talaga", "holy grail", "hides my acne"
]

negative_keywords = [
    "bad", "worst", "terrible", "awful", "hate", "dislike", "disappointed",
    "disappointing", "poor", "waste", "overpriced", "expensive", "cakey",
    "creasy", "crease", "oxidize", "oxidized", "patchy", "streaky",
    "heavy", "greasy", "oily", "dry", "drying", "irritated", "irritating",
    "broke out", "breakout", "pimple", "allergy", "allergic",
    "not worth", "would not recommend", "return", "fake", "expired",
    "hindi maganda", "hindi okay", "hindi ok", "hindi sulit", "mahal",
    "pangit", "basura", "hindi bet", "nakaka-off", "nakakainis",
    "nakaka-irritate", "sumabog", "nag-oxidize", "hindi maayos",
    "peke", "bigo", "nabigo", "hindi worth it", "sayang", "ayaw ko",
    "di maganda", "di okay", "di ok", "di sulit", "di bet", "messy"
]

# ── Pre-compile regex patterns ────────────────────────────────────────────────
brands_patterns:   List[Tuple[str, re.Pattern[str]]] = [(b, re.compile(r'\b' + re.escape(b) + r'\b')) for b in brands_list]
category_patterns: List[Tuple[str, re.Pattern[str]]] = [(c, re.compile(r'\b' + re.escape(c) + r'\b')) for c in makeup_categories]
platform_patterns: List[Tuple[str, re.Pattern[str]]] = [(p, re.compile(r'\b' + re.escape(p) + r'\b')) for p in ["tiktok", "shopee", "lazada"]]
positive_patterns: List[Tuple[str, re.Pattern[str]]] = [(k, re.compile(r'\b' + re.escape(k) + r'\b')) for k in positive_keywords]
negative_patterns: List[Tuple[str, re.Pattern[str]]] = [(k, re.compile(r'\b' + re.escape(k) + r'\b')) for k in negative_keywords]

# ── Helper Functions ──────────────────────────────────────────────────────────
def detect_brands(text):
    if not text:
        return None
    t = text.lower()
    found = [b for b, pat in brands_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_category(text):
    if not text:
        return None
    t = text.lower()
    found = [c for c, pat in category_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_platforms(text):
    if not text:
        return None
    t = text.lower()
    found = [p for p, pat in platform_patterns if pat.search(t)]
    return ", ".join(found) if found else None

def detect_sentiment(text):
    if not text:
        return "neutral"
    t = text.lower()
    pos = sum(1 for k, pat in positive_patterns if pat.search(t))  # ← fixed
    neg = sum(1 for k, pat in negative_patterns if pat.search(t))  # ← fixed
    if pos > neg:   return "positive"
    elif neg > pos: return "negative"
    else:           return "neutral"

def detect_sentiment_keywords(text):
    if not text:
        return None
    t = text.lower()
    found_pos = [k for k, pat in positive_patterns if pat.search(t)]  # ← fixed
    found_neg = [k for k, pat in negative_patterns if pat.search(t)]  # ← fixed
    all_found = found_pos + found_neg
    return ", ".join(all_found) if all_found else None

def ts_to_date(utc_ts):
    return datetime.utcfromtimestamp(utc_ts).strftime("%Y-%m-%d")

detect_brands_udf             = udf(detect_brands,              StringType())
detect_category_udf           = udf(detect_category,            StringType())
detect_platforms_udf          = udf(detect_platforms,           StringType())
detect_sentiment_udf          = udf(detect_sentiment,           StringType())
detect_sentiment_keywords_udf = udf(detect_sentiment_keywords,  StringType())

# ── Scrape last 7 days using new() ────────────────────────────────────────────
subreddit     = reddit.subreddit(SUBREDDIT)
posts_rows    = []
comments_rows = []

print("🔍 Scraping new posts...")

for submission in subreddit.new(limit=500):
    post_date = datetime.fromtimestamp(submission.created_utc, tz=timezone.utc)

    if post_date < DATE_FROM:
        break
    if post_date > DATE_TO:
        continue

    flair = submission.link_flair_text
    if not flair or FLAIR_FILTER not in flair.lower():
        continue

    post_content = submission.title
    if submission.selftext:
        post_content += " " + submission.selftext

    posts_rows.append((
        submission.id,
        post_content,
        submission.url,
        int(submission.score),
        int(submission.num_comments),
        ts_to_date(submission.created_utc),
    ))

    submission.comments.replace_more(limit=0)
    for comment in submission.comments.list():
        comments_rows.append((
            submission.id,
            comment.id,
            comment.body,
            int(comment.score),
            ts_to_date(comment.created_utc),
        ))

print(f"✅ Found {len(posts_rows)} new posts, {len(comments_rows)} comments.")

# ── Schemas ───────────────────────────────────────────────────────────────────
posts_schema = StructType([
    StructField("post_id",            StringType(),  False),
    StructField("post_content",       StringType(),  True),
    StructField("post_url",           StringType(),  True),
    StructField("number_of_upvotes",  IntegerType(), True),
    StructField("number_of_comments", IntegerType(), True),
    StructField("date",               StringType(),  True),
])

comments_schema = StructType([
    StructField("post_id",         StringType(),  False),
    StructField("comment_id",      StringType(),  False),
    StructField("comment_body",    StringType(),  True),
    StructField("comment_upvotes", IntegerType(), True),
    StructField("date",            StringType(),  True),
])

# ── Append only new posts (deduplicate by post_id) ────────────────────────────
posts_df    = spark.createDataFrame(posts_rows,    schema=posts_schema)
comments_df = spark.createDataFrame(comments_rows, schema=comments_schema)

posts_df    = posts_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))
comments_df = comments_df.withColumn("date", to_date(col("date"), "yyyy-MM-dd"))

existing_post_ids    = [r.post_id    for r in spark.table(POSTS_TABLE).select("post_id").collect()]
existing_comment_ids = [r.comment_id for r in spark.table(COMMENTS_TABLE).select("comment_id").collect()]

posts_df    = posts_df.filter(~col("post_id").isin(existing_post_ids))
comments_df = comments_df.filter(~col("comment_id").isin(existing_comment_ids))

posts_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(POSTS_TABLE)

comments_df.write \
    .format("delta") \
    .mode("append") \
    .option("mergeSchema", "true") \
    .saveAsTable(COMMENTS_TABLE)

print(f"✅ New posts appended:    {posts_df.count()} rows")
print(f"✅ New comments appended: {comments_df.count()} rows")

# ── Rebuild sentence tables from full table ───────────────────────────────────
print("🔍 Rebuilding sentence tables...")

post_sentences_df = spark.table(POSTS_TABLE) \
    .select("post_id", "post_content") \
    .withColumn("post_content", regexp_replace(col("post_content"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("post_content"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("post_content") \
    .dropDuplicates(["post_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("keywords_found",     detect_sentiment_keywords_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull()) \
    .filter(col("category_mentioned").isNotNull()) \
    .filter(trim(col("category_mentioned")) != "") \
    .filter(~col("brand_found").contains(","))

post_sentences_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(POST_SENT_TABLE)

comment_sentences_df = spark.table(COMMENTS_TABLE) \
    .select("post_id", "comment_id", "comment_body") \
    .withColumn("comment_body", regexp_replace(col("comment_body"), r"\n+", " ")) \
    .withColumn("sentence", explode(split(col("comment_body"), r"(?<=[.!?])\s+"))) \
    .withColumn("sentence", trim(col("sentence"))) \
    .filter(length(col("sentence")) > 5) \
    .drop("comment_body") \
    .dropDuplicates(["comment_id", "sentence"]) \
    .withColumn("brand_found",        detect_brands_udf(col("sentence"))) \
    .withColumn("category_mentioned", detect_category_udf(col("sentence"))) \
    .withColumn("platform_mentioned", detect_platforms_udf(col("sentence"))) \
    .withColumn("sentiment",          detect_sentiment_udf(col("sentence"))) \
    .withColumn("keywords_found",     detect_sentiment_keywords_udf(col("sentence"))) \
    .withColumn("date",               to_date(lit(datetime.now(timezone.utc).strftime("%Y-%m-%d")), "yyyy-MM-dd")) \
    .filter(col("brand_found").isNotNull()) \
    .filter(col("category_mentioned").isNotNull()) \
    .filter(trim(col("category_mentioned")) != "") \
    .filter(~col("brand_found").contains(","))

comment_sentences_df.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(COMM_SENT_TABLE)

print(f"✅ Post sentences:    {post_sentences_df.count()} rows")
print(f"✅ Comment sentences: {comment_sentences_df.count()} rows")

print("\n🎉 Weekly pipeline complete!")
print(f"   Total posts:             {spark.table(POSTS_TABLE).count()}")
print(f"   Total comments:          {spark.table(COMMENTS_TABLE).count()}")
print(f"   Total post sentences:    {spark.table(POST_SENT_TABLE).count()}")
print(f"   Total comment sentences: {spark.table(COMM_SENT_TABLE).count()}")

In [0]:
display(spark.table("beauty_analytics.post_brands_mentioned"))
